# 05 - Post-Hoc Robustness Checks

Runs the six post-hoc checks reported in Appendix I using the retained outputs from Notebooks 01 to 03. These checks do not alter the primary models or Chapter 4 results.

## 1. Imports and configuration

Loads the completed experimental run and the fixed settings used by the post-hoc checks.

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import time

import joblib
import numpy as np
import torch
import torch.nn as nn

from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset, Subset


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
def find_project_root() -> Path:
    """Find the repository root from the data/nsynth folder."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "nsynth").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/nsynth. Start Jupyter inside the repository."
    )


PROJECT_ROOT = find_project_root()
LATEST_RUN_PATH = PROJECT_ROOT / "results_final" / "latest_run.json"
if not LATEST_RUN_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 before Notebook 05.")
with LATEST_RUN_PATH.open("r", encoding="utf-8") as f:
    RUN_TAG_FROM_POINTER = json.load(f)["run_tag"]
RESULTS_DIR = PROJECT_ROOT / "results_final" / f"run_{RUN_TAG_FROM_POINTER}"
CACHE_ROOT = PROJECT_ROOT / "cache_final"
SWEEP_DIR = PROJECT_ROOT / "results_sweep_final"
CONTROL_DIR = PROJECT_ROOT / "results_nonrecurrent_control"

OUTPUT_DIR = PROJECT_ROOT / "results_robustness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POSTHOC_RESULTS_PATH = OUTPUT_DIR / "posthoc_results.json"


# ---------------------------------------------------------------------
# Fixed seeds are used for repeatable sampling and model recovery in the
# recorded environment. Exact floating-point results may still vary across
# hardware and software environments.
# ---------------------------------------------------------------------
SEED = 42
PROJECTION_SEEDS = (0,)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------
# Locked configuration, read rather than restated
# ---------------------------------------------------------------------
with open(SWEEP_DIR / "selected_reservoir_config.json") as f:
    SELECTION_MANIFEST = json.load(f)

if SELECTION_MANIFEST.get("selection_status") != "locked":
    raise RuntimeError("The reservoir-selection manifest is not locked.")

with open(RESULTS_DIR / "results_summary_final.json") as f:
    RESULTS = json.load(f)

RESERVOIR_CFG = SELECTION_MANIFEST["reservoir_cfg"]
RESERVOIR_UNITS = int(RESERVOIR_CFG["units"])
FEATURE_CFG = SELECTION_MANIFEST["feature_cfg"]
FEATURE_TAG = SELECTION_MANIFEST["feature_tag"]
SCALER_TAG = SELECTION_MANIFEST["scaler_tag"]
STATE_TAG = RESULTS["config"]["state_tag"]
RUN_TAG = RESULTS["config"]["run_tag"]

# ---------------------------------------------------------------------
# Verify that Notebook 02 belongs to the locked Notebook 01 selection.
# ---------------------------------------------------------------------

primary_cfg = RESULTS["config"]

expected_run_dir_name = f"run_{RUN_TAG}"
if RESULTS_DIR.name != expected_run_dir_name:
    raise RuntimeError(
        "Results directory and results_summary_final.json disagree: "
        f"{RESULTS_DIR.name} != {expected_run_dir_name}"
    )

if primary_cfg["feature_tag"] != FEATURE_TAG:
    raise RuntimeError(
        "Notebook 02 feature tag does not match the locked "
        "Notebook 01 manifest."
    )

if primary_cfg["scaler_tag"] != SCALER_TAG:
    raise RuntimeError(
        "Notebook 02 scaler tag does not match the locked "
        "Notebook 01 manifest."
    )

if primary_cfg["feature_cfg"] != FEATURE_CFG:
    raise RuntimeError(
        "Notebook 02 feature configuration does not match the "
        "locked Notebook 01 manifest."
    )

if primary_cfg["reservoir_cfg"] != RESERVOIR_CFG:
    raise RuntimeError(
        "Notebook 02 reservoir configuration does not match the "
        "locked Notebook 01 manifest."
    )

if int(primary_cfg["seed"]) != int(
    SELECTION_MANIFEST["reservoir_seed"]
):
    raise RuntimeError(
        "Notebook 02 reservoir seed does not match the locked "
        "Notebook 01 manifest."
    )

MAX_T = int(FEATURE_CFG["max_t"])
FEAT_DIM = int(FEATURE_CFG["n_mels"]) * (3 if FEATURE_CFG["use_deltas"] else 1)
N_CLASSES = 10
ATTACK_FRAMES = 20

MEL_CACHE_DIR = CACHE_ROOT / f"mel_{FEATURE_TAG}"
STATE_CACHE_DIR = CACHE_ROOT / f"states_{STATE_TAG}"
STATE_SCALER_PATH = STATE_CACHE_DIR / f"state_scaler_{STATE_TAG}.joblib"
FEATURE_SCALER_PATH = MEL_CACHE_DIR / f"feature_scaler_{SCALER_TAG}.joblib"

# ---------------------------------------------------------------------
# Ridge and readout settings, taken from the locked manifest so that the
# refits below use the identical grid, solver and tolerances.
# ---------------------------------------------------------------------
RIDGE_ALPHAS = list(SELECTION_MANIFEST["search_design"]["ridge_alphas"])
RIDGE_CLASS_WEIGHT = "balanced"
RIDGE_SOLVER = SELECTION_MANIFEST["search_design"]["ridge_solver"]
RIDGE_TOL = float(SELECTION_MANIFEST["search_design"]["ridge_tol"])
RIDGE_MAX_ITER = int(SELECTION_MANIFEST["search_design"]["ridge_max_iter"])
RIDGE_TIE_ATOL = 1e-12
VALIDATION_METRIC = "macro_f1"

HIDDEN_DIM = 128
DROPOUT = 0.30
BATCH_SIZE = 512
NUM_WORKERS = int(os.environ.get("ESN_NUM_WORKERS", "0"))
ROBUSTNESS_SEEDS = (0, 1, 2, 3, 4)

# Training-subset size for the fit diagnostic in check 3. A subsample is used
FIT_DIAGNOSTIC_N = 20_000

# Target dimensionality for check 5, matching the pooled acoustic representation of 3 x FEAT_DIM.
PROJECTION_DIM = FEAT_DIM * 3

print(f"Project root:   {PROJECT_ROOT}")
print(f"Run:            {RESULTS_DIR.name}")
print(f"State cache:    {STATE_CACHE_DIR.name}")
print(f"Control results:{CONTROL_DIR}")
print(f"Output:         {OUTPUT_DIR}")
print(f"Device:         {DEVICE}")
print(f"Reservoir:      {RESERVOIR_CFG}")
print(f"Ridge grid:     {len(RIDGE_ALPHAS)} values, "
      f"{min(RIDGE_ALPHAS):.0e} to {max(RIDGE_ALPHAS):.0e}")
print(f"Projection dim: {PROJECTION_DIM}")

Project root:   /home/olliechandler/ESN-NSYNTH
Run:            run_35d8eed08b
State cache:    states_242d8f6835
Control results:/home/olliechandler/ESN-NSYNTH/results_nonrecurrent_control
Output:         /home/olliechandler/ESN-NSYNTH/results_robustness
Device:         cuda
Reservoir:      {'units': 1000, 'sr': 0.99, 'lr': 0.05, 'input_scaling': 0.2, 'rc_connectivity': 0.1}
Ridge grid:     17 values, 1e-10 to 1e+04
Projection dim: 576


## 2. Required artefacts

Loads the feature caches, state caches, control outputs and five matched neural checkpoint pairs.

In [2]:
train_mel_X = MEL_CACHE_DIR / "X_train.npy"
valid_mel_X = MEL_CACHE_DIR / "X_valid.npy"
test_mel_X = MEL_CACHE_DIR / "X_test.npy"
train_y_path = MEL_CACHE_DIR / "y_train.npy"
valid_y_path = MEL_CACHE_DIR / "y_valid.npy"
test_y_path = MEL_CACHE_DIR / "y_test.npy"

state_paths = {
    split: STATE_CACHE_DIR / f"X_states_{split}.npy"
    for split in ("train", "valid", "test")
}

required = [
    train_mel_X, valid_mel_X, test_mel_X,
    train_y_path, valid_y_path, test_y_path,
    STATE_SCALER_PATH, FEATURE_SCALER_PATH,
    *state_paths.values(),
]

missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required artefacts are missing. Run notebooks 01 and 02 first:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

CONTROL_RESULTS_PATH = (
    PROJECT_ROOT
    / "results_nonrecurrent_control"
    / "results_nonrecurrent_control.json"
)

if not CONTROL_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Missing Notebook 03 result file: "
        f"{CONTROL_RESULTS_PATH}"
    )

with CONTROL_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    control_results = json.load(file)

control_cfg = control_results["config"]
CONTROL_TAG = control_cfg["control_tag"]

# ---------------------------------------------------------------------
# Verify that Notebook 03 belongs to the same retained experiment.
# ---------------------------------------------------------------------

if control_cfg["feature_tag"] != primary_cfg["feature_tag"]:
    raise RuntimeError(
        "Notebook 03 control feature tag does not match Notebook 02."
    )

if control_cfg["scaler_tag"] != primary_cfg["scaler_tag"]:
    raise RuntimeError(
        "Notebook 03 control scaler tag does not match Notebook 02."
    )

if control_cfg["base_reservoir_cfg"] != primary_cfg["reservoir_cfg"]:
    raise RuntimeError(
        "Notebook 03 control was built from a different reservoir "
        "configuration."
    )

if int(control_cfg["seed"]) != int(primary_cfg["seed"]):
    raise RuntimeError(
        "Notebook 03 control seed does not match Notebook 02."
    )

for split_key in ("n_train", "n_valid", "n_test"):
    if int(control_cfg[split_key]) != int(primary_cfg[split_key]):
        raise RuntimeError(
            f"Notebook 03 {split_key} does not match Notebook 02."
        )

CONTROL_CACHE_DIR = (
    CACHE_ROOT
    / f"states_nonrecurrent_control_{CONTROL_TAG}"
)

if not CONTROL_CACHE_DIR.is_dir():
    raise FileNotFoundError(
        "The control cache identified by Notebook 03 is missing: "
        f"{CONTROL_CACHE_DIR}"
    )
control_pooled_paths = {
    split: CONTROL_CACHE_DIR / f"X_pooled_{split}.npy"
    for split in ("train", "valid", "test")
}
missing_control = [
    path for path in control_pooled_paths.values() if not path.exists()
]
if missing_control:
    raise FileNotFoundError(
        "Control pooled caches are missing. Run notebook 03:\n"
        + "\n".join(f"  - {path}" for path in missing_control)
    )

# Neural checkpoints are discovered rather than restated.
checkpoints = {path.stem[len("best_"):]: path
               for path in sorted(RESULTS_DIR.glob("best_*.pt"))}
if not checkpoints:
    raise FileNotFoundError(
        f"No neural checkpoints found in {RESULTS_DIR}. Checks 2 and 3 "
        "require the best_*.pt files written by notebook 02."
    )

y_train = np.asarray(np.load(train_y_path, mmap_mode="r"), dtype=np.int64)
y_valid = np.asarray(np.load(valid_y_path, mmap_mode="r"), dtype=np.int64)
y_test = np.asarray(np.load(test_y_path, mmap_mode="r"), dtype=np.int64)

print(f"Control tag:        {CONTROL_TAG}")
print(f"Checkpoints found:  {len(checkpoints)}")
print(f"Splits:             {len(y_train):,} / {len(y_valid):,} / "
      f"{len(y_test):,}")

# Notebook 02 names the primary pair `mean_seed_0` and `attention_seed_0`,
# and the robustness sweep prefixes its tags, giving `robust_mean_seed_1`
# and so on. Both schemes are searched so that all five seeds are found.
def find_pair(seed: int):
    for prefix in ("", "robust_"):
        mean_tag = f"{prefix}mean_seed_{seed}"
        attention_tag = f"{prefix}attention_seed_{seed}"
        if mean_tag in checkpoints and attention_tag in checkpoints:
            return mean_tag, attention_tag
    return None


primary_pairs = [
    pair for pair in (find_pair(seed) for seed in ROBUSTNESS_SEEDS)
    if pair is not None
]
found_seeds = [int(pair[0].split("_")[-1]) for pair in primary_pairs]
print(f"Matched seed pairs: {found_seeds}")

if not primary_pairs:
    raise FileNotFoundError(
        "No matched mean/attention checkpoint pairs were found. Checks 2 "
        "and 3 require them. Available: "
        + ", ".join(sorted(checkpoints)[:8]) + " ..."
    )
if len(primary_pairs) != len(ROBUSTNESS_SEEDS):
    missing_seeds = sorted(
        set(ROBUSTNESS_SEEDS) - set(found_seeds)
    )
    raise FileNotFoundError(
        "Notebook 05 requires all five matched mean/attention "
        "checkpoint pairs for Checks 2 and 3. "
        f"Missing seeds: {missing_seeds}"
    )

Control tag:        5d3b246266
Checkpoints found:  22
Splits:             283,704 / 12,678 / 4,096
Matched seed pairs: [0, 1, 2, 3, 4]


## 3. Shared procedures

Defines the evaluation, masking, Ridge and pooling helpers reused by the six checks.

In [3]:
def classification_metrics(labels, predictions) -> dict:
    return {
        "acc": float(accuracy_score(labels, predictions)),
        "macro_f1": float(
            f1_score(labels, predictions, average="macro")
        ),
        "balanced_acc": float(
            balanced_accuracy_score(labels, predictions)
        ),
    }


def select_ridge_candidate(rows: list) -> dict:
    """Select on validation macro F1, preferring stronger regularisation
    when validation predictions tie."""
    best_score = max(row[VALIDATION_METRIC] for row in rows)
    tied = [
        row for row in rows
        if np.isclose(
            row[VALIDATION_METRIC],
            best_score,
            rtol=0.0,
            atol=RIDGE_TIE_ATOL,
        )
    ]
    return max(tied, key=lambda row: row["alpha"])


def ridge_sweep(Xtr, ytr, Xva, yva, Xte, yte, label: str) -> dict:
    """Fit the matched Ridge procedure across the complete alpha grid."""
    summary_scaler = StandardScaler().fit(Xtr)
    Xtr_s = summary_scaler.transform(Xtr)
    Xva_s = summary_scaler.transform(Xva)
    Xte_s = summary_scaler.transform(Xte)

    rows, predictions = [], {}
    started = time.time()

    for alpha in RIDGE_ALPHAS:
        classifier = RidgeClassifier(
            alpha=alpha,
            class_weight=RIDGE_CLASS_WEIGHT,
            solver=RIDGE_SOLVER,
            tol=RIDGE_TOL,
            max_iter=RIDGE_MAX_ITER,
        )
        classifier.fit(Xtr_s, ytr)
        validation_prediction = classifier.predict(Xva_s)
        rows.append({
            "alpha": float(alpha),
            **classification_metrics(yva, validation_prediction),
        })
        predictions[float(alpha)] = classifier.predict(Xte_s)

    selected = select_ridge_candidate(rows)
    test_prediction = predictions[selected["alpha"]]
    test_metrics = classification_metrics(yte, test_prediction)

    grid = [row[VALIDATION_METRIC] for row in rows]
    print(
        f"  {label:38s} alpha={selected['alpha']:<8.4g} "
        f"test macro F1={test_metrics['macro_f1'] * 100:6.2f}%  "
        f"(grid spread {(max(grid) - min(grid)) * 100:5.2f}pp, "
        f"{time.time() - started:.0f}s)"
    )

    del Xtr_s, Xva_s, Xte_s, summary_scaler
    gc.collect()

    return {
        "label": label,
        "selected_alpha": float(selected["alpha"]),
        "validation": {k: v for k, v in selected.items() if k != "alpha"},
        "test": test_metrics,
        "validation_sweep": rows,
        "validation_grid_spread": float(max(grid) - min(grid)),
        "test_predictions": test_prediction,
    }


class StateCacheDataset(Dataset):
    """Identical to the notebook 02 dataset, including the copy that avoids
    a tensor backed by a read-only memmap view."""

    def __init__(self, X_path: Path, y_path: Path):
        self.X_path = str(X_path)
        self.y_path = str(y_path)
        self.X = None
        self.y = None
        self.n = np.load(self.y_path, mmap_mode="r").shape[0]

    def _open(self):
        if self.X is None:
            self.X = np.load(self.X_path, mmap_mode="r")
        if self.y is None:
            self.y = np.load(self.y_path, mmap_mode="r")

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        self._open()
        x = torch.from_numpy(
            np.asarray(self.X[idx], dtype=np.float32).copy()
        )
        y = torch.tensor(int(self.y[idx]), dtype=torch.long)
        return x, y


class TemporalPoolingReadout(nn.Module):
    """The notebook 02 readout, reproduced so that saved checkpoints load."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        n_classes: int,
        dropout: float,
        pooling: str,
        keep_mask=None,
    ):
        super().__init__()
        if pooling not in {"mean", "attention"}:
            raise ValueError("pooling must be 'mean' or 'attention'")

        self.pooling = pooling
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

        if keep_mask is None:
            keep_mask = np.ones(MAX_T, dtype=bool)
        keep_mask = np.asarray(keep_mask, dtype=bool)
        if keep_mask.shape != (MAX_T,) or not keep_mask.any():
            raise ValueError(
                "keep_mask must retain at least one of MAX_T frames"
            )
        self.register_buffer(
            "keep_mask", torch.tensor(keep_mask, dtype=torch.bool)
        )

        if pooling == "mean":
            for parameter in self.score.parameters():
                parameter.requires_grad = False

    def forward(self, x, return_attention: bool = False):
        h = torch.tanh(self.proj(x))
        keep = self.keep_mask.unsqueeze(0)

        if self.pooling == "attention":
            scores = self.score(h).squeeze(-1)
            scores = scores.masked_fill(~keep, float("-inf"))
            weights = torch.softmax(scores, dim=1)
        else:
            base = self.keep_mask.to(dtype=h.dtype)
            base = base / base.sum()
            weights = base.unsqueeze(0).expand(h.shape[0], -1)

        pooled = torch.sum(h * weights.unsqueeze(-1), dim=1)
        logits = self.classifier(self.dropout(pooled))
        return (logits, weights) if return_attention else logits


def load_checkpoint(tag: str, pooling: str, keep_mask=None):
    """Load a saved readout, optionally under a mask it was not trained with.

    The keep_mask buffer is part of the saved state, so it is restored after
    loading rather than before. This is what makes test-time masking a
    change to the fitted model's inputs rather than a retrained model.
    """
    model = TemporalPoolingReadout(
        input_dim=RESERVOIR_UNITS,
        hidden_dim=HIDDEN_DIM,
        n_classes=N_CLASSES,
        dropout=DROPOUT,
        pooling=pooling,
    ).to(DEVICE)

    state = torch.load(checkpoints[tag], map_location=DEVICE)
    model.load_state_dict(state)

    if keep_mask is not None:
        keep_mask = np.asarray(keep_mask, dtype=bool)
        model.keep_mask = torch.tensor(
            keep_mask, dtype=torch.bool, device=DEVICE
        )

    model.eval()
    return model


@torch.no_grad()
def evaluate_model(model, loader) -> dict:
    model.eval()
    all_preds, all_labels = [], []
    for X, y in loader:
        X = X.to(DEVICE, non_blocking=True)
        all_preds.append(model(X).argmax(dim=1).cpu().numpy())
        all_labels.append(y.numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return {**classification_metrics(labels, preds), "preds": preds}


def summarise_sequence_cache(
    X_path: Path,
    indices=None,
    scaler=None,
    chunk_size: int = 1_024,
) -> np.ndarray:
    """Mean, elementwise maximum and final frame, matching notebook 02."""
    X = np.load(X_path, mmap_mode="r")
    if indices is None:
        indices = np.arange(len(X), dtype=np.int64)
    else:
        indices = np.asarray(indices, dtype=np.int64)

    feat_dim = X.shape[-1]
    out = np.empty((len(indices), feat_dim * 3), dtype=np.float32)

    for start in range(0, len(indices), chunk_size):
        end = min(start + chunk_size, len(indices))
        A = np.asarray(X[indices[start:end]], dtype=np.float32)
        if scaler is not None:
            A = scaler.transform(
                A.reshape(-1, feat_dim)
            ).reshape(A.shape)
        out[start:end] = np.concatenate(
            [A.mean(axis=1), A.max(axis=1), A[:, -1, :]], axis=1
        )

    del X
    gc.collect()
    return out


def stratified_subset(y: np.ndarray, n: int, seed: int) -> np.ndarray:
    if n >= len(y):
        return np.arange(len(y), dtype=np.int64)
    splitter = StratifiedShuffleSplit(
        n_splits=1, train_size=n, random_state=seed
    )
    indices, _ = next(splitter.split(np.zeros(len(y)), y))
    return np.sort(indices.astype(np.int64))


posthoc = {
    "run_tag": RUN_TAG,
    "state_tag": STATE_TAG,
    "control_tag": CONTROL_TAG,
    "reservoir_cfg": RESERVOIR_CFG,
    "seed": SEED,
}
print("Shared procedures defined.")

Shared procedures defined.


## 4. Check 1 - state scaler verification

Recreates the reservoir-state standardiser check reported in Appendix I.5.

In [4]:
import reservoirpy as rpy
from reservoirpy.nodes import Reservoir

os.environ["RESERVOIRPY_VERBOSITY"] = "0"
rpy.set_seed(SEED)

feature_scaler = joblib.load(FEATURE_SCALER_PATH)
deployed_state_scaler = joblib.load(STATE_SCALER_PATH)


def build_reservoir():
    """The notebook 02 construction, from the locked configuration."""
    reservoir = Reservoir(**RESERVOIR_CFG, seed=SEED)
    reservoir.run(np.zeros((3, FEAT_DIM), dtype=np.float32))
    return reservoir


def run_reservoir_on_feature(
    feature_sequence, fitted_feature_scaler, fitted_reservoir
):
    feature_scaled = fitted_feature_scaler.transform(
        feature_sequence.astype(np.float32)
    )
    fitted_reservoir.reset()
    return np.asarray(
        fitted_reservoir.run(feature_scaled), dtype=np.float32
    )


scaler_sample_n = int(
    SELECTION_MANIFEST["scaler_cfg"]["scaler_n_examples"]
)
X_mel_train = np.load(train_mel_X, mmap_mode="r")

# Identical sampling to notebook 02: default_rng(SEED).choice, unsorted,
# then one partial_fit per recording in that order.
rng = np.random.default_rng(SEED)
scaler_indices = rng.choice(
    len(X_mel_train),
    size=min(scaler_sample_n, len(X_mel_train)),
    replace=False,
)

reservoir = build_reservoir()
refit_state_scaler = StandardScaler()
started = time.time()

for position, index in enumerate(scaler_indices):
    states = run_reservoir_on_feature(
        np.asarray(X_mel_train[index], dtype=np.float32),
        feature_scaler,
        reservoir,
    )
    refit_state_scaler.partial_fit(states)
    if (position + 1) % 500 == 0 or position + 1 == len(scaler_indices):
        print(f"  scaler {position + 1:,}/{len(scaler_indices):,} "
              f"| {time.time() - started:.1f}s")

mean_difference = float(
    np.max(np.abs(refit_state_scaler.mean_ - deployed_state_scaler.mean_))
)
scale_difference = float(
    np.max(np.abs(refit_state_scaler.scale_ - deployed_state_scaler.scale_))
)

print()
print("State-standardiser verification")
print(f"  Sample size:                    {len(scaler_indices):,} recordings")
print(f"  Max absolute difference, means: {mean_difference:.3e}")
print(f"  Max absolute difference, SDs:   {scale_difference:.3e}")

if max(mean_difference, scale_difference) > 1e-6:
    print(
        "\n  NOTE: the refit does not reproduce the deployed standardiser. "
        "Check that the locked reservoir configuration and feature scaler "
        "match those used by notebook 02 before reporting this result."
    )

posthoc["scaler_verification"] = {
    "sample_recordings": int(len(scaler_indices)),
    "max_abs_difference_mean": mean_difference,
    "max_abs_difference_scale": scale_difference,
    "reproduces_deployed": bool(
        max(mean_difference, scale_difference) <= 1e-6
    ),
}

del X_mel_train, refit_state_scaler, reservoir
gc.collect()

  scaler 500/3,000 | 4.2s
  scaler 1,000/3,000 | 8.4s
  scaler 1,500/3,000 | 12.6s
  scaler 2,000/3,000 | 16.8s
  scaler 2,500/3,000 | 21.0s
  scaler 3,000/3,000 | 25.2s

State-standardiser verification
  Sample size:                    3,000 recordings
  Max absolute difference, means: 0.000e+00
  Max absolute difference, SDs:   0.000e+00


0

## 5. Check 2 - test-time state masking

Evaluates the saved neural readouts with temporal state masking and no retraining. See Appendix I.4.

In [5]:
early_keep_mask = np.ones(MAX_T, dtype=bool)
early_keep_mask[:ATTACK_FRAMES] = False

central_start = (MAX_T - ATTACK_FRAMES) // 2
central_keep_mask = np.ones(MAX_T, dtype=bool)
central_keep_mask[central_start:central_start + ATTACK_FRAMES] = False

early_only_keep_mask = np.zeros(MAX_T, dtype=bool)
early_only_keep_mask[:ATTACK_FRAMES] = True

mask_conditions = {
    "full": np.ones(MAX_T, dtype=bool),
    "mask_early": early_keep_mask,
    "mask_control": central_keep_mask,
    "keep_early_only": early_only_keep_mask,
}

print(f"Central control window: states "
      f"{central_start + 1}-{central_start + ATTACK_FRAMES}")

test_dataset = StateCacheDataset(state_paths["test"], test_y_path)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

test_time_masking = {}
started = time.time()

for mean_tag, attention_tag in primary_pairs:
    seed = int(mean_tag.split("_")[-1])
    test_time_masking[seed] = {}

    for pooling, tag in (("mean", mean_tag), ("attention", attention_tag)):
        test_time_masking[seed][pooling] = {}
        for condition, keep_mask in mask_conditions.items():
            model = load_checkpoint(tag, pooling, keep_mask=keep_mask)
            result = evaluate_model(model, test_loader)
            test_time_masking[seed][pooling][condition] = {
                key: result[key]
                for key in ("acc", "macro_f1", "balanced_acc")
            }
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

print(f"Evaluated in {time.time() - started:.0f}s\n")

header = (
    f"{'Condition':22s}{'mean F1':>10s}{'delta':>9s}"
    f"{'attn F1':>10s}{'delta':>9s}"
)
print("Test-time masking, no retraining, averaged across "
      f"{len(primary_pairs)} seeds")
print(header)

summary_rows = {}
for condition in mask_conditions:
    row = {}
    for pooling in ("mean", "attention"):
        values = [
            test_time_masking[s][pooling][condition]["macro_f1"] * 100
            for s in test_time_masking
        ]
        full = [
            test_time_masking[s][pooling]["full"]["macro_f1"] * 100
            for s in test_time_masking
        ]
        row[pooling] = {
            "macro_f1_mean": float(np.mean(values)),
            "macro_f1_std": float(np.std(values, ddof=0)),
            "delta_vs_full": float(np.mean(values) - np.mean(full)),
        }
    summary_rows[condition] = row
    print(
        f"{condition:22s}"
        f"{row['mean']['macro_f1_mean']:10.2f}"
        f"{row['mean']['delta_vs_full']:+9.2f}"
        f"{row['attention']['macro_f1_mean']:10.2f}"
        f"{row['attention']['delta_vs_full']:+9.2f}"
    )

posthoc["test_time_masking"] = {
    "per_seed": test_time_masking,
    "summary": summary_rows,
    "central_window_start_state": int(central_start + 1),
}

Central control window: states 53-72
Evaluated in 83s

Test-time masking, no retraining, averaged across 5 seeds
Condition                mean F1    delta   attn F1    delta
full                       69.46    +0.00     67.68    +0.00
mask_early                 65.09    -4.37     50.85   -16.83
mask_control               68.56    -0.90     67.37    -0.32
keep_early_only            33.54   -35.92     40.21   -27.47


## 6. Check 3 - training and validation fit

Compares training and validation performance for the selected neural checkpoints. See Appendix I.3.

In [ ]:
fit_indices = stratified_subset(y_train, FIT_DIAGNOSTIC_N, SEED)

train_subset = Subset(
    StateCacheDataset(state_paths["train"], train_y_path),
    fit_indices.tolist(),
)
train_eval_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)
valid_loader = DataLoader(
    StateCacheDataset(state_paths["valid"], valid_y_path),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

print(f"Training subset: {len(fit_indices):,} recordings "
      f"({len(fit_indices) / len(y_train) * 100:.1f}% of training)")

fit_diagnostic = {}
started = time.time()

for mean_tag, attention_tag in primary_pairs:
    seed = int(mean_tag.split("_")[-1])
    fit_diagnostic[seed] = {}
    for pooling, tag in (("mean", mean_tag), ("attention", attention_tag)):
        model = load_checkpoint(tag, pooling)
        train_result = evaluate_model(model, train_eval_loader)
        valid_result = evaluate_model(model, valid_loader)
        fit_diagnostic[seed][pooling] = {
            "train_macro_f1": train_result["macro_f1"],
            "valid_macro_f1": valid_result["macro_f1"],
            "gap": train_result["macro_f1"] - valid_result["macro_f1"],
        }
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

print(f"Evaluated in {time.time() - started:.0f}s\n")
print(f"{'Seed':>6s}{'mean train':>13s}{'mean valid':>13s}{'gap':>8s}"
      f"{'attn train':>13s}{'attn valid':>13s}{'gap':>8s}")

for seed in sorted(fit_diagnostic):
    entry = fit_diagnostic[seed]
    print(
        f"{seed:6d}"
        f"{entry['mean']['train_macro_f1'] * 100:13.2f}"
        f"{entry['mean']['valid_macro_f1'] * 100:13.2f}"
        f"{entry['mean']['gap'] * 100:8.2f}"
        f"{entry['attention']['train_macro_f1'] * 100:13.2f}"
        f"{entry['attention']['valid_macro_f1'] * 100:13.2f}"
        f"{entry['attention']['gap'] * 100:8.2f}"
    )

summary = {}
for pooling in ("mean", "attention"):
    for field in ("train_macro_f1", "valid_macro_f1", "gap"):
        values = [
            fit_diagnostic[s][pooling][field] for s in fit_diagnostic
        ]
        summary[f"{pooling}_{field}_mean"] = float(np.mean(values))
        summary[f"{pooling}_{field}_std"] = float(np.std(values, ddof=0))

print()
print(f"Mean training gap:      {summary['mean_gap_mean'] * 100:+.2f}pp")
print(f"Attention training gap: {summary['attention_gap_mean'] * 100:+.2f}pp")

if (
    summary["attention_train_macro_f1_mean"]
    > summary["mean_train_macro_f1_mean"]
    and summary["attention_valid_macro_f1_mean"]
    < summary["mean_valid_macro_f1_mean"]
):
    reading = (
    "Attention achieves higher training performance but lower validation "
    "performance, giving a larger train-validation gap in this diagnostic."
)
elif (
    summary["attention_train_macro_f1_mean"]
    < summary["mean_train_macro_f1_mean"]
    and summary["attention_valid_macro_f1_mean"]
    < summary["mean_valid_macro_f1_mean"]
):
    reading = (
        "Attention is lower on both training and validation, which "
        "indicates underfitting or an optimisation difficulty rather than "
        "a generalisation gap."
    )
else:
    reading = (
        "The pattern is mixed. Report the mechanism as unresolved rather "
        "than adopting either account."
    )

print(f"\nReading: {reading}")

posthoc["fit_diagnostic"] = {
    "training_subset_n": int(len(fit_indices)),
    "per_seed": fit_diagnostic,
    "summary": summary,
    "reading": reading,
}

Training subset: 20,000 recordings (7.0% of training)
Evaluated in 193s

  Seed   mean train   mean valid     gap   attn train   attn valid     gap
     0        82.88        68.10   14.78        87.22        66.80   20.42
     1        84.93        68.83   16.10        86.98        67.92   19.06
     2        86.02        69.53   16.49        88.95        68.32   20.64
     3        85.78        69.31   16.47        84.72        65.69   19.04
     4        85.24        69.16   16.08        86.62        66.56   20.06

Mean training gap:      +15.98pp
Attention training gap: +19.84pp

Reading: Attention fits the training subset better but generalises worse, which supports the regularisation account.


## 7. Check 4 - recurrent and non-recurrent pooling blocks

valuates the non-recurrent mean, maximum and final-state pooling blocks. The matched recurrent comparison is completed in Section 11. See Appendix I.1.

In [7]:
control_pooled = {
    split: np.load(control_pooled_paths[split], mmap_mode="r")
    for split in ("train", "valid", "test")
}
pooled_width = control_pooled["train"].shape[1]
block_width = pooled_width // 3

if pooled_width != RESERVOIR_UNITS * 3:
    raise RuntimeError(
        f"Unexpected pooled control width {pooled_width}, expected "
        f"{RESERVOIR_UNITS * 3}."
    )

blocks = {
    "mean_block": slice(0, block_width),
    "max_block": slice(block_width, 2 * block_width),
    "final_block": slice(2 * block_width, 3 * block_width),
}

print(f"Pooled control: {pooled_width} features, "
      f"{block_width} per block\n")

block_results = {}

for name, columns in blocks.items():
    result = ridge_sweep(
        np.asarray(control_pooled["train"][:, columns], dtype=np.float32),
        y_train,
        np.asarray(control_pooled["valid"][:, columns], dtype=np.float32),
        y_valid,
        np.asarray(control_pooled["test"][:, columns], dtype=np.float32),
        y_test,
        label=f"control / {name}",
    )
    result.pop("test_predictions")
    block_results[name] = result
    gc.collect()

print()
print(f"{'Block':16s}{'dims':>7s}{'alpha':>10s}{'test F1':>10s}"
      f"{'grid spread':>13s}")
for name, result in block_results.items():
    dims = block_width if name != "concatenated" else pooled_width
    print(
        f"{name:16s}{dims:7d}{result['selected_alpha']:10.4g}"
        f"{result['test']['macro_f1'] * 100:10.2f}"
        f"{result['validation_grid_spread'] * 100:13.2f}"
    )

posthoc["control_block_ablation"] = block_results

del control_pooled
gc.collect()

Pooled control: 3000 features, 1000 per block

  control / mean_block                   alpha=0.1      test macro F1= 60.03%  (grid spread  4.95pp, 4188s)
  control / max_block                    alpha=1        test macro F1= 53.35%  (grid spread  0.37pp, 1500s)
  control / final_block                  alpha=1000     test macro F1= 18.78%  (grid spread  0.41pp, 3988s)

Block              dims     alpha   test F1  grid spread
mean_block         1000       0.1     60.03         4.95
max_block          1000         1     53.35         0.37
final_block        1000      1000     18.78         0.41


0

## 8. Check 5 - dimensionality-matched RQ1

Projects the pooled ESN and control representations to the acoustic dimensionality before repeating the RQ1 Ridge comparison. See Appendix I.2.

In [8]:
acoustic_reference = RESULTS["rq1"]["acoustic_ridge"]["test"]["macro_f1"]
esn_reference = RESULTS["rq1"]["esn_state_ridge"]["test"]["macro_f1"]

print("Pooling reservoir states from the cache...")
started = time.time()

esn_pooled = {
    split: summarise_sequence_cache(
        state_paths[split]
    )
    for split in ("train", "valid", "test")
}

print(
    f"  pooled in {time.time() - started:.0f}s, "
    f"{esn_pooled['train'].shape[1]} features\n"
)

control_pooled = {
    split: np.asarray(
        np.load(
            control_pooled_paths[split],
            mmap_mode="r",
        ),
        dtype=np.float32,
    )
    for split in ("train", "valid", "test")
}

print(
    "Reported baselines: "
    f"acoustic {acoustic_reference * 100:.2f}%, "
    f"ESN {esn_reference * 100:.2f}%\n"
)

projection_results = {}

for projection_seed in PROJECTION_SEEDS:
    # Gaussian random projection scaled by 1/sqrt(k).
    # The same matrix is applied to both representations.
    generator = np.random.default_rng(
        projection_seed
    )

    projection = generator.normal(
        loc=0.0,
        scale=1.0 / np.sqrt(PROJECTION_DIM),
        size=(
            RESERVOIR_UNITS * 3,
            PROJECTION_DIM,
        ),
    ).astype(np.float32)

    print(
        f"Projection seed {projection_seed}: "
        f"{RESERVOIR_UNITS * 3} -> "
        f"{PROJECTION_DIM}"
    )

    seed_results = {}

    for name, pooled in (
        ("esn", esn_pooled),
        ("control", control_pooled),
    ):
        result = ridge_sweep(
            pooled["train"] @ projection,
            y_train,
            pooled["valid"] @ projection,
            y_valid,
            pooled["test"] @ projection,
            y_test,
            label=(
                f"{name} / projected seed "
                f"{projection_seed}"
            ),
        )

        result.pop("test_predictions")
        seed_results[name] = result
        gc.collect()

    projection_results[
        projection_seed
    ] = seed_results

    del projection
    gc.collect()

print()
print(
    f"{'Seed':>6s}"
    f"{'ESN projected':>16s}"
    f"{'vs reported':>13s}"
    f"{'control projected':>19s}"
    f"{'vs acoustic':>13s}"
)

for (
    projection_seed,
    seed_results,
) in projection_results.items():
    esn_f1 = (
        seed_results["esn"]["test"]["macro_f1"]
        * 100
    )
    control_f1 = (
        seed_results["control"]["test"]["macro_f1"]
        * 100
    )

    print(
        f"{projection_seed:6d}"
        f"{esn_f1:16.2f}"
        f"{esn_f1 - esn_reference * 100:+13.2f}"
        f"{control_f1:19.2f}"
        f"{control_f1 - acoustic_reference * 100:+13.2f}"
    )

esn_projected = [
    projection_results[
        seed
    ]["esn"]["test"]["macro_f1"]
    * 100
    for seed in PROJECTION_SEEDS
]

retained = (
    np.mean(esn_projected)
    - acoustic_reference * 100
) / (
    esn_reference * 100
    - acoustic_reference * 100
)

print()
print(
    "ESN projected macro F1: "
    f"{np.mean(esn_projected):.2f} "
    f"± {np.std(esn_projected, ddof=0):.2f}%"
)
print(
    "Proportion of the reported ESN advantage over "
    "the acoustic baseline retained after "
    "dimensionality matching: "
    f"{retained * 100:.1f}%"
)

posthoc["dimensionality_matched"] = {
    "projection_dim": int(PROJECTION_DIM),
    "projection_seeds": list(
        PROJECTION_SEEDS
    ),
    "acoustic_reference_macro_f1": (
        acoustic_reference
    ),
    "esn_reference_macro_f1": (
        esn_reference
    ),
    "per_seed": {
        str(key): value
        for key, value
        in projection_results.items()
    },
    "esn_projected_mean": float(
        np.mean(esn_projected)
    ),
    "esn_projected_std": float(
        np.std(
            esn_projected,
            ddof=0,
        )
    ),
    "advantage_retained_fraction": float(
        retained
    ),
}

del esn_pooled, control_pooled
gc.collect()

Pooling reservoir states from the cache...
  pooled in 286s, 3000 features

Reported baselines: acoustic 45.79%, ESN 66.65%

Projection seed 0: 3000 -> 576
  esn / projected seed 0                 alpha=100      test macro F1= 57.47%  (grid spread  2.03pp, 1261s)
  control / projected seed 0             alpha=10       test macro F1= 52.96%  (grid spread  0.86pp, 808s)

  Seed   ESN projected  vs reported  control projected  vs acoustic
     0           57.47        -9.18              52.96        +7.17

ESN projected macro F1: 57.47 ± 0.00%
Proportion of the reported ESN advantage over the acoustic baseline retained after dimensionality matching: 56.0%


0

## 9. Check 6 - temporal state structure

Compares early and later cached reservoir-state structure. See Appendix I.6.

In [9]:
# Initial-state / early-trajectory diagnostic

from sklearn.model_selection import train_test_split

DIAG_N = 500
DIAG_SEED = 42
chunk_size = 256

X_states_train = np.load(state_paths["train"], mmap_mode="r")

assert X_states_train.shape[0] == len(y_train)
assert X_states_train.shape[1:] == (MAX_T, RESERVOIR_UNITS)
assert np.all((y_train >= 0) & (y_train < N_CLASSES))

# Fixed class-stratified training subset.
# This keeps the diagnostic separate from final test evaluation.
all_idx = np.arange(len(y_train))
diag_idx, _ = train_test_split(
    all_idx,
    train_size=DIAG_N,
    stratify=y_train,
    random_state=DIAG_SEED,
)

diag_idx = np.sort(diag_idx)
diag_labels = y_train[diag_idx]
n_diag = len(diag_idx)

frame_sum = np.zeros(
    (MAX_T, RESERVOIR_UNITS), dtype=np.float64
)
frame_sumsq = np.zeros_like(frame_sum)

class_sum = np.zeros(
    (N_CLASSES, MAX_T, RESERVOIR_UNITS), dtype=np.float64
)
class_count = np.zeros(N_CLASSES, dtype=np.float64)

for start in range(0, n_diag, chunk_size):
    end = min(start + chunk_size, n_diag)
    idx = diag_idx[start:end]

    block = np.asarray(
        X_states_train[idx], dtype=np.float64
    )
    labels = diag_labels[start:end]

    frame_sum += block.sum(axis=0)
    frame_sumsq += (block ** 2).sum(axis=0)

    for label in np.unique(labels):
        mask = labels == label
        class_sum[label] += block[mask].sum(axis=0)
        class_count[label] += int(mask.sum())

grand_mean = frame_sum / n_diag

total_variance = (
    frame_sumsq / n_diag - grand_mean ** 2
)
total_variance = np.maximum(total_variance, 0.0)

between = np.zeros_like(total_variance)

for label in range(N_CLASSES):
    if class_count[label] == 0:
        continue

    class_mean = (
        class_sum[label] / class_count[label]
    )

    between += (
        class_count[label] / n_diag
    ) * (class_mean - grand_mean) ** 2

within = np.maximum(
    total_variance - between, 0.0
)

# Same feature-wise B/W convention used in Appendix F.3,
# calculated here separately for each temporal position.
eps = 1e-12
usable = within > eps

ratio_per_frame = np.array([
    float(np.mean(
        between[t][usable[t]]
        / within[t][usable[t]]
    ))
    if usable[t].any()
    else np.nan
    for t in range(MAX_T)
])

# Across-recording SD for each unit, averaged over units.
sd_per_frame = np.sqrt(
    total_variance
).mean(axis=1)

leak_rate = float(RESERVOIR_CFG["lr"])


direct_leak_carryover = (
    (1.0 - leak_rate)
    ** np.arange(1, MAX_T + 1)
)

early = slice(0, ATTACK_FRAMES)
late = slice(ATTACK_FRAMES, MAX_T)

print(
    f"Diagnostic sample: {n_diag} training recordings "
    f"(stratified, seed {DIAG_SEED})"
)

print(
    f"Leak rate: {leak_rate:.3f}"
)

print(
    f"Direct leak-path carry-over after "
    f"{ATTACK_FRAMES} updates: "
    f"{direct_leak_carryover[ATTACK_FRAMES - 1] * 100:.1f}%"
)

print(
    "Note: this is not the fraction of the "
    "initialisation transient remaining; recurrent "
    "propagation is not included.\n"
)

print(
    f"{'Region':22s}"
    f"{'across-recording SD':>22s}"
    f"{'between/within':>17s}"
)

print(
    f"{'First 20 states':22s}"
    f"{sd_per_frame[early].mean():22.4f}"
    f"{np.nanmean(ratio_per_frame[early]):17.4f}"
)

print(
    f"{'Remaining 105 states':22s}"
    f"{sd_per_frame[late].mean():22.4f}"
    f"{np.nanmean(ratio_per_frame[late]):17.4f}"
)

print(
    f"{'Whole trajectory':22s}"
    f"{sd_per_frame.mean():22.4f}"
    f"{np.nanmean(ratio_per_frame):17.4f}"
)

print(
    f"\n{'Frame':>6s}"
    f"{'time (s)':>10s}"
    f"{'SD':>10s}"
    f"{'B/W':>10s}"
    f"{'direct leak':>14s}"
)

for t in (0, 4, 9, 19, 39, 59, 124):
    print(
        f"{t + 1:6d}"
        f"{(t + 1) * 512 / 16000:10.2f}"
        f"{sd_per_frame[t]:10.4f}"
        f"{ratio_per_frame[t]:10.4f}"
        f"{direct_leak_carryover[t] * 100:13.1f}%"
    )

early_sd_ratio = float(
    sd_per_frame[early].mean()
    / sd_per_frame[late].mean()
)

early_bw_ratio = float(
    np.nanmean(ratio_per_frame[early])
    / np.nanmean(ratio_per_frame[late])
)

print(
    f"\nEarly relative to later states:"
    f" SD ratio = {early_sd_ratio:.3f},"
    f" B/W ratio = {early_bw_ratio:.3f}"
)

print(
    "\nInterpretation: these statistics describe whether "
    "the early reservoir states retain across-recording "
    "variation and instrument-family structure relative "
    "to later states. They do not directly measure the "
    "influence of reservoir initialisation and should not "
    "be treated as a washout or memory estimate."
)

posthoc["initialisation_diagnostic"] = {
    "sample_partition": "train",
    "sample_size": int(n_diag),
    "sample_seed": DIAG_SEED,
    "sample_stratified": True,

    "leak_rate": leak_rate,

    "direct_leak_carryover_at_early_boundary": float(
        direct_leak_carryover[
            ATTACK_FRAMES - 1
        ]
    ),

    "direct_leak_carryover_note": (
        "Explicit (1-alpha) identity-path contribution "
        "only; not total initial-state influence or "
        "reservoir memory."
    ),

    "across_recording_sd_per_frame": (
        sd_per_frame.tolist()
    ),

    "between_within_ratio_per_frame": (
        ratio_per_frame.tolist()
    ),

    "early_sd_mean": float(
        sd_per_frame[early].mean()
    ),

    "late_sd_mean": float(
        sd_per_frame[late].mean()
    ),

    "early_between_within_mean": float(
        np.nanmean(
            ratio_per_frame[early]
        )
    ),

    "late_between_within_mean": float(
        np.nanmean(
            ratio_per_frame[late]
        )
    ),

    "early_sd_ratio": early_sd_ratio,
    "early_between_within_ratio": early_bw_ratio,

    "interpretation_scope": (
        "Comparison of early and later standardised "
        "cached state structure. Does not directly demonstrate "
        "reservoir initialisation effects."
    ),
}

del (
    X_states_train,
    frame_sum,
    frame_sumsq,
    class_sum,
)
gc.collect()

Diagnostic sample: 500 training recordings (stratified, seed 42)
Leak rate: 0.050
Direct leak-path carry-over after 20 updates: 35.8%
Note: this is not the fraction of the initialisation transient remaining; recurrent propagation is not included.

Region                   across-recording SD   between/within
First 20 states                       0.5481           0.2118
Remaining 105 states                  1.0157           0.1403
Whole trajectory                      0.9409           0.1518

 Frame  time (s)        SD       B/W   direct leak
     1      0.03    0.1043    0.3117         95.0%
     5      0.16    0.4712    0.2904         77.4%
    10      0.32    0.5867    0.2045         59.9%
    20      0.64    0.7431    0.1322         35.8%
    40      1.28    0.9751    0.1201         12.9%
    60      1.92    1.0957    0.1305          4.6%
   125      4.00    0.7750    0.1422          0.2%

Early relative to later states: SD ratio = 0.540, B/W ratio = 1.510

Interpretation: these sta

11

## 10. Save post-hoc results

Writes the six checks to `results_robustness/posthoc_results.json` without modifying the primary experiment outputs.

In [10]:
output_path = POSTHOC_RESULTS_PATH
temp_path = output_path.with_name("posthoc_results.json.tmp")

def to_serialisable(obj):
    if isinstance(obj, dict):
        return {str(k): to_serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serialisable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


with open(temp_path, "w") as f:
    json.dump(to_serialisable(posthoc), f, indent=2)
temp_path.replace(output_path)

checks = {
    "scaler_verification": "scaler_verification" in posthoc,
    "test_time_masking": "test_time_masking" in posthoc,
    "fit_diagnostic": "fit_diagnostic" in posthoc,
    "control_block_ablation": "control_block_ablation" in posthoc,
    "dimensionality_matched": "dimensionality_matched" in posthoc,
    "initialisation_diagnostic": "initialisation_diagnostic" in posthoc,
    "results_written": output_path.exists(),
}

print("Completion checks")
for name, passed in checks.items():
    print(f"  {name:26s}: {passed}")

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(f"Post-hoc checks incomplete. Failed: {failed}")

print(f"\nSaved: {output_path}")
print("\nPost-hoc robustness checks complete.")

Completion checks
  scaler_verification       : True
  test_time_masking         : True
  fit_diagnostic            : True
  control_block_ablation    : True
  dimensionality_matched    : True
  initialisation_diagnostic : True
  results_written           : True

Saved: /home/olliechandler/ESN-NSYNTH/results_robustness/posthoc_results.json

Post-hoc robustness checks complete.


## 11. Matched recurrent pooling-block analysis

Completes the pooling-block comparison reported in Appendix I.1 by evaluating the retained ESN mean, maximum and final-state blocks using the same Ridge procedure applied to the non-recurrent control. The comparison is written to the existing post-hoc results file and does not alter the primary RQ1 results.

In [1]:
# =====================================================================
# POST-HOC EXTENSION - MATCHED ESN POOLING-BLOCK ANALYSIS
# =====================================================================
# Standalone cell. Safe to run after the original post-hoc notebook has
# completed and posthoc_results.json has been written.
#
# Purpose:
#   Compare the mean, maximum and final-state pooling blocks of the
#   retained ESN representation against the corresponding non-recurrent
#   control blocks.
#
# This DOES NOT:
#   - regenerate acoustic features
#   - rerun the reservoir
#   - rebuild state caches
#   - retrain neural readouts
#   - alter the primary RQ1 results
#
# It reads the retained standardised ESN state caches written by
# Notebook 02 and fits only three matched Ridge classifiers.
# =====================================================================

from pathlib import Path
import gc
import json
import time

import numpy as np

from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------
# 1. Locate project and retained experiment
# ---------------------------------------------------------------------
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "nsynth").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/nsynth. Start Jupyter inside the repository."
    )


PROJECT_ROOT = find_project_root()

LATEST_RUN_PATH = (
    PROJECT_ROOT / "results_final" / "latest_run.json"
)
SELECTION_PATH = (
    PROJECT_ROOT
    / "results_sweep_final"
    / "selected_reservoir_config.json"
)
POSTHOC_RESULTS_PATH = (
    PROJECT_ROOT
    / "results_robustness"
    / "posthoc_results.json"
)

for path in (
    LATEST_RUN_PATH,
    SELECTION_PATH,
    POSTHOC_RESULTS_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


with LATEST_RUN_PATH.open("r", encoding="utf-8") as f:
    run_tag = json.load(f)["run_tag"]

RESULTS_DIR = (
    PROJECT_ROOT
    / "results_final"
    / f"run_{run_tag}"
)

with SELECTION_PATH.open("r", encoding="utf-8") as f:
    selection = json.load(f)

with (
    RESULTS_DIR / "results_summary_final.json"
).open("r", encoding="utf-8") as f:
    results = json.load(f)

with POSTHOC_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    posthoc = json.load(f)


# ---------------------------------------------------------------------
# 2. Recover locked configuration and caches
# ---------------------------------------------------------------------
RESERVOIR_UNITS = int(
    selection["reservoir_cfg"]["units"]
)
STATE_TAG = results["config"]["state_tag"]

STATE_CACHE_DIR = (
    PROJECT_ROOT
    / "cache_final"
    / f"states_{STATE_TAG}"
)

FEATURE_TAG = selection["feature_tag"]
MEL_CACHE_DIR = (
    PROJECT_ROOT
    / "cache_final"
    / f"mel_{FEATURE_TAG}"
)

state_paths = {
    split: (
        STATE_CACHE_DIR
        / f"X_states_{split}.npy"
    )
    for split in ("train", "valid", "test")
}

label_paths = {
    split: (
        MEL_CACHE_DIR
        / f"y_{split}.npy"
    )
    for split in ("train", "valid", "test")
}

required = [
    *state_paths.values(),
    *label_paths.values(),
]

missing = [
    path for path in required
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required retained caches are missing:\n"
        + "\n".join(
            f"  - {path}" for path in missing
        )
    )


y_train = np.asarray(
    np.load(
        label_paths["train"],
        mmap_mode="r",
    ),
    dtype=np.int64,
)

y_valid = np.asarray(
    np.load(
        label_paths["valid"],
        mmap_mode="r",
    ),
    dtype=np.int64,
)

y_test = np.asarray(
    np.load(
        label_paths["test"],
        mmap_mode="r",
    ),
    dtype=np.int64,
)


# ---------------------------------------------------------------------
# 3. Recover the exact Ridge procedure
# ---------------------------------------------------------------------
RIDGE_ALPHAS = list(
    selection["search_design"]["ridge_alphas"]
)
RIDGE_CLASS_WEIGHT = "balanced"
RIDGE_SOLVER = (
    selection["search_design"]["ridge_solver"]
)
RIDGE_TOL = float(
    selection["search_design"]["ridge_tol"]
)
RIDGE_MAX_ITER = int(
    selection["search_design"]["ridge_max_iter"]
)

RIDGE_TIE_ATOL = 1e-12
VALIDATION_METRIC = "macro_f1"


def classification_metrics(
    labels,
    predictions,
):
    return {
        "acc": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
            )
        ),
        "balanced_acc": float(
            balanced_accuracy_score(
                labels,
                predictions,
            )
        ),
    }


def select_ridge_candidate(rows):
    best_score = max(
        row[VALIDATION_METRIC]
        for row in rows
    )

    tied = [
        row
        for row in rows
        if np.isclose(
            row[VALIDATION_METRIC],
            best_score,
            rtol=0.0,
            atol=RIDGE_TIE_ATOL,
        )
    ]

    # Same rule as the primary experiment:
    # prefer stronger regularisation when
    # validation predictions tie.
    return max(
        tied,
        key=lambda row: row["alpha"],
    )


def ridge_sweep(
    Xtr,
    ytr,
    Xva,
    yva,
    Xte,
    yte,
    label,
):
    summary_scaler = StandardScaler().fit(
        Xtr
    )

    Xtr_s = summary_scaler.transform(
        Xtr
    )
    Xva_s = summary_scaler.transform(
        Xva
    )
    Xte_s = summary_scaler.transform(
        Xte
    )

    rows = []
    predictions = {}

    started = time.time()

    for alpha in RIDGE_ALPHAS:
        classifier = RidgeClassifier(
            alpha=alpha,
            class_weight=RIDGE_CLASS_WEIGHT,
            solver=RIDGE_SOLVER,
            tol=RIDGE_TOL,
            max_iter=RIDGE_MAX_ITER,
        )

        classifier.fit(
            Xtr_s,
            ytr,
        )

        validation_prediction = (
            classifier.predict(Xva_s)
        )

        rows.append({
            "alpha": float(alpha),
            **classification_metrics(
                yva,
                validation_prediction,
            ),
        })

        predictions[float(alpha)] = (
            classifier.predict(Xte_s)
        )

    selected = select_ridge_candidate(
        rows
    )

    test_prediction = predictions[
        selected["alpha"]
    ]

    test_metrics = (
        classification_metrics(
            yte,
            test_prediction,
        )
    )

    grid = [
        row[VALIDATION_METRIC]
        for row in rows
    ]

    print(
        f"  {label:28s} "
        f"alpha={selected['alpha']:<8.4g} "
        f"test F1="
        f"{test_metrics['macro_f1'] * 100:6.2f}% "
        f"| spread="
        f"{(max(grid)-min(grid))*100:5.2f}pp "
        f"| {time.time()-started:.0f}s"
    )

    del (
        Xtr_s,
        Xva_s,
        Xte_s,
        summary_scaler,
    )
    gc.collect()

    return {
        "label": label,
        "selected_alpha": float(
            selected["alpha"]
        ),
        "validation": {
            key: value
            for key, value
            in selected.items()
            if key != "alpha"
        },
        "test": test_metrics,
        "validation_sweep": rows,
        "validation_grid_spread": float(
            max(grid) - min(grid)
        ),
    }


# ---------------------------------------------------------------------
# 4. Pool retained ESN trajectories
# ---------------------------------------------------------------------
# Notebook 02's state caches are already standardised.
# Therefore no state scaler is applied here.
#
# Output order is exactly:
#   [temporal mean | elementwise maximum | final state]
# ---------------------------------------------------------------------
def summarise_sequence_cache(
    X_path,
    chunk_size=1024,
):
    X = np.load(
        X_path,
        mmap_mode="r",
    )

    n_examples = X.shape[0]
    feat_dim = X.shape[-1]

    out = np.empty(
        (
            n_examples,
            feat_dim * 3,
        ),
        dtype=np.float32,
    )

    for start in range(
        0,
        n_examples,
        chunk_size,
    ):
        end = min(
            start + chunk_size,
            n_examples,
        )

        A = np.asarray(
            X[start:end],
            dtype=np.float32,
        )

        out[start:end] = np.concatenate(
            [
                A.mean(axis=1),
                A.max(axis=1),
                A[:, -1, :],
            ],
            axis=1,
        )

    del X
    gc.collect()

    return out


print("=" * 68)
print(
    "Matched ESN pooling-block analysis"
)
print("=" * 68)

print(
    f"Run tag:   {run_tag}"
)
print(
    f"State tag: {STATE_TAG}"
)
print(
    f"Splits:    "
    f"{len(y_train):,} / "
    f"{len(y_valid):,} / "
    f"{len(y_test):,}"
)

print(
    "\nPooling retained ESN "
    "state caches..."
)

pool_started = time.time()

esn_pooled = {
    split: summarise_sequence_cache(
        state_paths[split]
    )
    for split in (
        "train",
        "valid",
        "test",
    )
}

pool_seconds = (
    time.time() - pool_started
)

pooled_width = (
    esn_pooled["train"].shape[1]
)
block_width = (
    pooled_width // 3
)

if (
    pooled_width
    != RESERVOIR_UNITS * 3
):
    raise RuntimeError(
        f"Unexpected pooled width "
        f"{pooled_width}; expected "
        f"{RESERVOIR_UNITS * 3}."
    )

print(
    f"Pooling complete in "
    f"{pool_seconds:.0f}s"
)
print(
    f"Pooled width: "
    f"{pooled_width}"
)
print(
    f"Block width:  "
    f"{block_width}"
)


# ---------------------------------------------------------------------
# 5. Fit only the three missing ESN block models
# ---------------------------------------------------------------------
blocks = {
    "mean_block": slice(
        0,
        block_width,
    ),
    "max_block": slice(
        block_width,
        2 * block_width,
    ),
    "final_block": slice(
        2 * block_width,
        3 * block_width,
    ),
}

print(
    "\nRunning matched Ridge "
    "sweeps..."
)

esn_block_results = {}

for (
    block_name,
    columns,
) in blocks.items():

    esn_block_results[
        block_name
    ] = ridge_sweep(
        np.asarray(
            esn_pooled["train"][
                :,
                columns,
            ],
            dtype=np.float32,
        ),
        y_train,

        np.asarray(
            esn_pooled["valid"][
                :,
                columns,
            ],
            dtype=np.float32,
        ),
        y_valid,

        np.asarray(
            esn_pooled["test"][
                :,
                columns,
            ],
            dtype=np.float32,
        ),
        y_test,

        label=(
            f"ESN / {block_name}"
        ),
    )

    gc.collect()


# ---------------------------------------------------------------------
# 6. Compare directly with retained control results
# ---------------------------------------------------------------------
if (
    "control_block_ablation"
    not in posthoc
):
    raise KeyError(
        "Existing posthoc_results.json "
        "does not contain "
        "'control_block_ablation'."
    )

control_blocks = (
    posthoc[
        "control_block_ablation"
    ]
)

esn_concat_f1 = float(
    results[
        "rq1"
    ][
        "esn_state_ridge"
    ][
        "test"
    ][
        "macro_f1"
    ]
)


with (
    PROJECT_ROOT
    / "results_nonrecurrent_control"
    / "results_nonrecurrent_control.json"
).open(
    "r",
    encoding="utf-8",
) as f:
    control_primary = json.load(f)

control_concat_f1 = float(
    control_primary[
        "test"
    ][
        "macro_f1"
    ]
)


print()
print("=" * 68)

print(
    f"{'Pooling':16s}"
    f"{'ESN F1':>10s}"
    f"{'Control F1':>13s}"
    f"{'Difference':>14s}"
)

print("-" * 68)

comparison = {}

for block_name in blocks:

    esn_f1 = (
        esn_block_results[
            block_name
        ][
            "test"
        ][
            "macro_f1"
        ]
        * 100
    )

    control_f1 = (
        control_blocks[
            block_name
        ][
            "test"
        ][
            "macro_f1"
        ]
        * 100
    )

    gap = esn_f1 - control_f1

    comparison[
        block_name
    ] = {
        "esn_macro_f1": float(
            esn_f1 / 100
        ),
        "control_macro_f1": float(
            control_f1 / 100
        ),
        "esn_minus_control_pp": float(
            gap
        ),
    }

    print(
        f"{block_name:16s}"
        f"{esn_f1:10.2f}"
        f"{control_f1:13.2f}"
        f"{gap:+14.2f}"
    )


concat_gap = (
    esn_concat_f1
    - control_concat_f1
) * 100

comparison[
    "concatenated"
] = {
    "esn_macro_f1": (
        esn_concat_f1
    ),
    "control_macro_f1": (
        control_concat_f1
    ),
    "esn_minus_control_pp": float(
        concat_gap
    ),
}

print(
    f"{'concatenated':16s}"
    f"{esn_concat_f1 * 100:10.2f}"
    f"{control_concat_f1 * 100:13.2f}"
    f"{concat_gap:+14.2f}"
)

print("=" * 68)


# ---------------------------------------------------------------------
# 7. Append to existing post-hoc JSON
# ---------------------------------------------------------------------
posthoc[
    "esn_block_ablation"
] = esn_block_results

posthoc[
    "pooling_block_comparison"
] = {
    "description": (
        "Matched mean, maximum and "
        "final-state Ridge comparisons "
        "for the retained recurrent ESN "
        "and non-recurrent control."
    ),
    "pooling_seconds": float(
        pool_seconds
    ),
    "blocks": comparison,
}


temp_path = (
    POSTHOC_RESULTS_PATH
    .with_name(
        "posthoc_results.json.tmp"
    )
)

with temp_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        posthoc,
        f,
        indent=2,
    )

temp_path.replace(
    POSTHOC_RESULTS_PATH
)


# ---------------------------------------------------------------------
# 8. Final checks
# ---------------------------------------------------------------------
with POSTHOC_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    verification = json.load(f)

assert (
    "esn_block_ablation"
    in verification
)
assert (
    "pooling_block_comparison"
    in verification
)

print(
    "\nSaved matched ESN "
    "block analysis to:"
)
print(
    POSTHOC_RESULTS_PATH
)
print(
    "\nESN pooling-block "
    "extension complete."
)

del esn_pooled
gc.collect()

Matched ESN pooling-block analysis
Run tag:   35d8eed08b
State tag: 242d8f6835
Splits:    283,704 / 12,678 / 4,096

Pooling retained ESN state caches...
Pooling complete in 124s
Pooled width: 3000
Block width:  1000

Running matched Ridge sweeps...
  ESN / mean_block             alpha=1        test F1= 59.99% | spread= 4.85pp | 4368s
  ESN / max_block              alpha=10       test F1= 60.90% | spread= 1.06pp | 1754s
  ESN / final_block            alpha=0.01     test F1= 54.00% | spread= 2.60pp | 4966s

Pooling             ESN F1   Control F1    Difference
--------------------------------------------------------------------
mean_block           59.99        60.03         -0.05
max_block            60.90        53.35         +7.55
final_block          54.00        18.78        +35.21
concatenated         66.65        62.13         +4.51

Saved matched ESN block analysis to:
/home/olliechandler/ESN-NSYNTH/results_robustness/posthoc_results.json

ESN pooling-block extension complete.


33